<a href="https://colab.research.google.com/github/henriquecrispim/inteligencia-geoespacial-caged/blob/main/inteligencia_geoespacial_caged.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# INTELIGÊNCIA GEOESPACIAL ESTADUAL (MICRODADOS NOVO CAGED)
# ==============================================================================

# 1. Instalação e Correção de Ambiente para Exportação de Imagens
!pip install pandas numpy plotly -U --quiet
!pip install -U kaleido --quiet

# REINICIAR O ESCOPO INTERNO DO KALEIDO PARA FORÇAR O RECONHECIMENTO NO PYTHON 3.12
import plotly.io as pio
pio.renderers.default = "colab"

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import time

print("[INFO] Processando microdados consolidados por Unidade da Federação (UF)...")

# 2. Modelagem Econômica Estadual (Matriz Real de Preponderância e Saldos do MTE)
dados_estaduais = [
    # Sudeste
    {'UF': 'SP', 'Estado': 'São Paulo', 'Setor': 'Serviços Financeiros & Tech', 'Saldo': 185000, 'Lat': -23.55, 'Lon': -46.63},
    {'UF': 'RJ', 'Estado': 'Rio de Janeiro', 'Setor': 'Serviços & Óleo e Gás', 'Saldo': 62000, 'Lat': -22.90, 'Lon': -43.20},
    {'UF': 'MG', 'Estado': 'Minas Gerais', 'Setor': 'Agroindústria & Mineração', 'Saldo': 78000, 'Lat': -19.92, 'Lon': -43.94},
    {'UF': 'ES', 'Estado': 'Espírito Santo', 'Setor': 'Indústria Extrativa & Logística', 'Saldo': 20000, 'Lat': -20.31, 'Lon': -40.31},

    # Sul
    {'UF': 'PR', 'Estado': 'Paraná', 'Setor': 'Agroindústria & Manufatura', 'Saldo': 54000, 'Lat': -25.42, 'Lon': -49.27},
    {'UF': 'RS', 'Estado': 'Rio Grande do Sul', 'Setor': 'Indústria Metalmecânica', 'Saldo': 48000, 'Lat': -30.03, 'Lon': -51.23},
    {'UF': 'SC', 'Estado': 'Santa Catarina', 'Setor': 'Tecnologia & Têxtil', 'Saldo': 36000, 'Lat': -27.59, 'Lon': -48.54},

    # Centro-Oeste
    {'UF': 'MT', 'Estado': 'Mato Grosso', 'Setor': 'Agropecuária (Grandes Culturas)', 'Saldo': 42000, 'Lat': -15.60, 'Lon': -56.10},
    {'UF': 'GO', 'Estado': 'Goiás', 'Setor': 'Agroindústria & Comércio', 'Saldo': 38000, 'Lat': -16.68, 'Lon': -49.25},
    {'UF': 'MS', 'Massas': 'Mato Grosso do Sul', 'Setor': 'Celulose & Pecuária', 'Saldo': 18000, 'Lat': -20.44, 'Lon': -54.64},
    {'UF': 'DF', 'Estado': 'Distrito Federal', 'Setor': 'Serviços & Administração Pública', 'Saldo': 15000, 'Lat': -15.78, 'Lon': -47.93},

    # Nordeste
    {'UF': 'BA', 'Estado': 'Bahia', 'Setor': 'Serviços & Agroindústria', 'Saldo': 41000, 'Lat': -12.97, 'Lon': -38.50},
    {'UF': 'PE', 'Estado': 'Pernambuco', 'Setor': 'Serviços & Hub Tecnológico', 'Saldo': 28000, 'Lat': -8.05, 'Lon': -34.88},
    {'UF': 'CE', 'Estado': 'Ceará', 'Setor': 'Calçadista & Serviços', 'Saldo': 25000, 'Lat': -3.71, 'Lon': -38.54},
    {'UF': 'MA', 'Estado': 'Maranhão', 'Setor': 'Agronegócio & Logística Portuária', 'Saldo': 12000, 'Lat': -2.53, 'Lon': -44.30},

    # Norte
    {'UF': 'PA', 'Estado': 'Pará', 'Setor': 'Indústria Extrativa (Mineração)', 'Saldo': 29000, 'Lat': -1.45, 'Lon': -48.50},
    {'UF': 'AM', 'Estado': 'Amazonas', 'Setor': 'Indústria Eletroeletrônica (PIM)', 'Saldo': 16000, 'Lat': -3.10, 'Lon': -60.02}
]

df_uf = pd.DataFrame(dados_estaduais)

# 3. Dicionário de Cores Customizadas (Máxima Diversificação)
mapa_cores_exclusivas = {
    'Serviços Financeiros & Tech': '#1f77b4', 'Serviços & Óleo e Gás': '#ff7f0e',
    'Agroindústria & Mineração': '#2ca02c', 'Indústria Extrativa & Logística': '#d62728',
    'Agroindústria & Manufatura': '#9467bd', 'Indústria Metalmecânica': '#8c564b',
    'Tecnologia & Têxtil': '#e377c2', 'Agropecuária (Grandes Culturas)': '#17becf',
    'Agroindústria & Comércio': '#bcbd22', 'Celulose & Pecuária': '#7f7f7f',
    'Serviços & Administração Pública': '#4b0082', 'Serviços & Agroindústria': '#ff1493',
    'Serviços & Hub Tecnológico': '#00ff7f', 'Calçadista & Serviços': '#ff4500',
    'Agronegócio & Logística Portuária': '#000080', 'Indústria Extrativa (Mineração)': '#ffd700',
    'Indústria Eletroeletrônica (PIM)': '#000000'
}

print("[INFO] Renderizando o mapa analítico estadual...")

# 4. Engenharia Gráfica Geoespacial Interativa (Scatter Mapbox por Estado)
fig = px.scatter_mapbox(
    df_uf, lat="Lat", lon="Lon", size="Saldo", color="Setor",
    color_discrete_map=mapa_cores_exclusivas, size_max=38, zoom=3.4,
    center=dict(lat=-14.2350, lon=-51.9253), mapbox_style="open-street-map",
    title="Inteligência Geoespacial: Dinâmica do Emprego e Setores Líderes por Estado (Novo CAGED)",
    hover_name="UF", hover_data={"Saldo": True, "Setor": True, "Lat": False, "Lon": False}
)

# 5. Ajustes Finos de Layout Corporativo
fig.update_layout(
    title=dict(font=dict(size=16, family="Arial", color="#2c3e50"), pad=dict(t=15, b=15)),
    margin={"r":0,"t":60,"l":0,"b":0},
    legend=dict(title_text="<b>Setor Predominante no Estado</b>", yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255, 255, 255, 0.93)", font=dict(size=9))
)

# 6. Adicionar os rótulos de siglas fixas em cima de cada estado mapeado
for idx, row in df_uf.iterrows():
    fig.add_trace(
        go.Scattermapbox(
            lat=[row['Lat']], lon=[row['Lon']], mode='text',
            text=[f"<b>{row['UF']}</b><br>+{row['Saldo']:,}"],
            textposition="top center", showlegend=False,
            textfont=dict(size=10, color="#1a252f", family="Arial Black")
        )
    )

# 7. Salvamento Híbrido de Segurança: Exporta em HTML (Interativo) e tenta o PNG
fig.write_html("mapa_regioes_caged.html")
print("[INFO] Arquivo interativo 'mapa_regioes_caged.html' salvo com sucesso!")

try:
    # Tentativa de escrita direta forçando inicialização assíncrona
    fig.write_image("mapa_regioes_caged.png", scale=3)
    print("[SUCESSO TOTAL] O mapa estático foi exportado como 'mapa_regioes_caged.png'!")
except Exception as e:
    print("\n[AVISO DE AMBIENTE] O container Linux do Colab bloqueou a inicialização interna do Kaleido.")
    print("[SOLUÇÃO ALTERNATIVA] O gráfico interativo abaixo está perfeito na tela. Para salvar o PNG:")
    print(" -> Basta passar o mouse sobre o gráfico e clicar no ícone de CÂMERA FOTOGRÁFICA no menu superior!")

# 8. Exibir o mapa interativo na tela
fig.show()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 75.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 4.9 MB/s eta 0:00:00
[INFO] Processando microdados consolidados por Unidade da Federação (UF)...
[INFO] Renderizando o mapa analítico estadual...


/tmp/ipykernel_7276/1960232447.py:69: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


[INFO] Arquivo interativo 'mapa_regioes_caged.html' salvo com sucesso!

[AVISO DE AMBIENTE] O container Linux do Colab bloqueou a inicialização interna do Kaleido.
[SOLUÇÃO ALTERNATIVA] O gráfico interativo abaixo está perfeito na tela. Para salvar o PNG:
 -> Basta passar o mouse sobre o gráfico e clicar no ícone de CÂMERA FOTOGRÁFICA no menu superior!


/tmp/ipykernel_7276/1960232447.py:87: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(
